In [0]:
%run /Workspace/Repos/Project_1705/Azure_Retail_Project/Project_Files/Functions/ACCESS_ADLS_GEN2_USING_SERVICE_PRINCIPALE 

In [0]:
%run /Workspace/Repos/Project_1705/Azure_Retail_Project/Project_Files/Functions/Silver_Layer_Functions

In [0]:
from datetime import datetime
currentdate = datetime.now().strftime("%Y%m%d")
print(currentdate)

In [0]:
dbutils.widgets.text("read_container_name","silver","Read Container")
read_container_name = dbutils.widgets.get("read_container_name")

In [0]:
dbutils.widgets.text("write_container_name","gold","Write Container")
write_container_name = dbutils.widgets.get("write_container_name")

In [0]:
dbutils.widgets.dropdown("table_name","sellers",["customers","products","sellers"],"Table Name")
table_name = dbutils.widgets.get("table_name")

In [0]:
read_base_path = f"abfss://{read_container_name}@retailstorage1881.dfs.core.windows.net"

In [0]:
silver_df = spark.read.format('delta').load(f"{read_base_path}/{table_name}/{currentdate}/")
silver_df.display()

In [0]:
write_base_path = f"abfss://{write_container_name}@retailstorage1881.dfs.core.windows.net"

In [0]:
from pyspark.sql.functions import current_date, lit
from delta.tables import DeltaTable

# Target Gold Path
target_path = f"{write_base_path}/{table_name}/"

# FIRST RUN
if not DeltaTable.isDeltaTable(spark, target_path):

    print("First Run Started")

    silver_df = silver_df \
        .withColumn("START_DATE", current_date()) \
        .withColumn("END_DATE", lit(None).cast("date")) \
        .withColumn("IS_CURRENT", lit("Y"))

    silver_df.write \
        .format("delta") \
        .mode("overwrite") \
        .save(target_path)

    print("Initial Load Completed")


# FUTURE RUNS
else:

    print("Incremental SCD2 Load Started")

    silver_df.createOrReplaceTempView("SOURCE")

    # Expire Old Records
    merge_query = f"""
    MERGE INTO delta.`{target_path}` TARGET
    USING SOURCE
    ON TARGET.seller_id  = SOURCE.seller_id 
    AND TARGET.IS_CURRENT = 'Y'

    WHEN MATCHED
    AND (
        COALESCE(TARGET.seller_zip_code_prefix,0) <> COALESCE(SOURCE.seller_zip_code_prefix,0)
        OR
        COALESCE(TARGET.seller_city,'') <> COALESCE(SOURCE.seller_city,'')
        OR
        COALESCE(TARGET.seller_state,'') <> COALESCE(SOURCE.seller_state,'')
    )

    THEN UPDATE SET
        TARGET.END_DATE = CURRENT_DATE(),
        TARGET.IS_CURRENT = 'N'
    """
    spark.sql(merge_query)


    # Insert New + Changed Records
    insert_query = f"""
    INSERT INTO delta.`{target_path}`

    (
        seller_id,
        seller_zip_code_prefix,
        seller_city,
        seller_state,
        START_DATE,
        END_DATE,
        IS_CURRENT
    )

    SELECT
        SOURCE.seller_id,
        SOURCE.seller_zip_code_prefix,
        SOURCE.seller_city,
        SOURCE.seller_state,
        CURRENT_DATE(),
        NULL,
        'Y'

    FROM SOURCE

    LEFT JOIN delta.`{target_path}` TARGET
    ON SOURCE.seller_id = TARGET.seller_id
    AND TARGET.IS_CURRENT = 'Y'

    WHERE TARGET.seller_id IS NULL

    OR (
        COALESCE(TARGET.seller_zip_code_prefix,0) <> COALESCE(SOURCE.seller_zip_code_prefix,0)
        OR
        COALESCE(TARGET.seller_city,'') <> COALESCE(SOURCE.seller_city,'')
        OR
        COALESCE(TARGET.seller_state,'') <> COALESCE(SOURCE.seller_state,'')
    )
    """

    spark.sql(insert_query)

    print("Incremental SCD2 Load Completed")